In [0]:
# 1. Define source location
source_path = "/Volumes/lakehouse/raw/olist_source"

# 2. List available files
display(dbutils.fs.ls(source_path))

# 3. Read orders
orders_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(f"{source_path}/olist_orders_dataset.csv")
)

# 4. Inspect data
orders_df.printSchema()

display(orders_df.limit(10))

# 5. Basic profiling
print(f"Rows: {orders_df.count()}")
print(f"Columns: {len(orders_df.columns)}")

path,name,size,modificationTime
dbfs:/Volumes/lakehouse/raw/olist_source/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1794403138000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1794403146000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1794403141000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1794403136000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1794403140000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1794403141000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1794403133000
dbfs:/Volumes/lakehouse/raw/olist_source/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1794403131000
dbfs:/Volumes/lakehouse/raw/olist_source/product_category_name_translation.csv,product_category_name_translation.csv,2613,1794403131000


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


Rows: 99441
Columns: 8


In [0]:
from pyspark.sql import functions as F

# 1. Basic data quality metrics
orders_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("distinct_order_ids"),
    F.sum(
        F.when(
            F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_order_ids")
).display()

# 2. Order status distribution
display(
    orders_df
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

total_rows,distinct_order_ids,missing_order_ids
99441,99441,0


order_status,count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
missing_metrics = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in orders_df.columns
]

display(orders_df.agg(*missing_metrics))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,0,0,160,1783,2965,0


In [0]:
duplicate_orders = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_orders)

print(f"Duplicate order IDs: {duplicate_orders.count()}")

order_id,count


Duplicate order IDs: 0


In [0]:
missing_data = (orders_df
    .groupBy("order_status")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.col("order_approved_at").isNull().cast("int")).alias("missing_approval"),
        F.sum(F.col("order_delivered_carrier_date").isNull().cast("int")).alias("missing_carrier_date"),
        F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("missing_customer_date")
    )
    .orderBy(F.desc("total_orders")))
    
missing_data.toPandas()

,order_status,total_orders,missing_approval,missing_carrier_date,missing_customer_date
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


In [0]:
potentially_invalid_rows = (
    orders_df
    .filter(
        (F.col("order_status") == "delivered") &
        (
            F.col("order_approved_at").isNull() |
            F.col("order_delivered_carrier_date").isNull() |
            F.col("order_delivered_customer_date").isNull()
        )
    )
    .select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

potentially_invalid_rows.toPandas()

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,None,2017-12-18 00:00:00
1,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,None,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
2,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,None,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
3,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,None,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
4,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,None,2018-07-16 00:00:00
5,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,None,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
6,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,None,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
7,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,None,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
8,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,None,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
9,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,None,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


In [0]:
timestamp_issues = orders_df.filter(
    (
        F.col("order_approved_at").isNotNull() &
        (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
    ) |
    (
        F.col("order_delivered_carrier_date").isNotNull() &
        F.col("order_approved_at").isNotNull() &
        (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
    ) |
    (
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_delivered_carrier_date").isNotNull() &
        (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
    )
)

display(timestamp_issues)
print(f"Records with impossible timestamp sequences: {timestamp_issues.count()}")

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
32e4fa9bb468884309b58b9348de70c3,e54367d4b00c5cb76d2dfe71b9bdb89c,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23 00:00:00
4df92d82d79c3b52c7138679fa9b07fc,ba0660bf3fffe505ee892e153a2fbd49,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06 00:00:00
16e38caa92e342c7780f68832f832d4d,d29935fdaf4a76f34653da3def4f2a24,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07 00:00:00
b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04 00:00:00
6051e6d3da9a50b7325cbe9c81025062,c25122206bfcb3ffa950706e1cb10f2b,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19 00:00:00


Records with impossible timestamp sequences: 1382


In [0]:
timestamp_checks = orders_df.agg(
    F.sum(
        F.when(
            F.col("order_approved_at").isNotNull() &
            (F.col("order_approved_at") < F.col("order_purchase_timestamp")),
            1
        ).otherwise(0)
    ).alias("approval_before_purchase"),

    F.sum(
        F.when(
            F.col("order_delivered_carrier_date").isNotNull() &
            F.col("order_approved_at").isNotNull() &
            (F.col("order_delivered_carrier_date") < F.col("order_approved_at")),
            1
        ).otherwise(0)
    ).alias("carrier_before_approval"),

    F.sum(
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_delivered_carrier_date").isNotNull() &
            (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date")),
            1
        ).otherwise(0)
    ).alias("customer_before_carrier")
)

timestamp_checks.toPandas()

,approval_before_purchase,carrier_before_approval,customer_before_carrier
0,0,1359,23


In [0]:
carrier_before_approval = orders_df.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    F.col("order_approved_at").isNotNull() &
    (
        F.col("order_delivered_carrier_date") <
        F.col("order_approved_at")
    )
)

display(
    carrier_before_approval
    .select(
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date"
    )
    .limit(20)
)

order_id,order_status,order_approved_at,order_delivered_carrier_date
dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00
688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14
58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00
412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00
56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00
32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00
4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00
16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00
b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00
6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00


In [0]:
display(
    carrier_before_approval
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

order_status,count
delivered,1350
shipped,9


In [0]:
display(
    carrier_before_approval
    .withColumn(
        "approval_month",
        F.date_format(
            F.to_timestamp("order_approved_at"),
            "yyyy-MM"
        )
    )
    .groupBy("approval_month")
    .count()
    .orderBy("approval_month")
)

approval_month,count
2017-03,1
2017-04,24
2017-05,14
2017-06,2
2017-07,3
2017-08,21
2017-09,13
2017-12,6
2018-01,1
2018-02,36


In [0]:
display(
    carrier_before_approval
    .withColumn(
        "hours_difference",
        (
            F.unix_timestamp("order_approved_at") -
            F.unix_timestamp("order_delivered_carrier_date")
        ) / 3600
    )
    .select("hours_difference")
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

summary,hours_difference
count,1359
mean,24.75198695936555
stddev,115.3077932651858
min,0.005833333333333334
25%,1.413888888888889
50%,17.16777777777778
75%,25.95888888888889
max,4109.256111111111


In [0]:
display(
    carrier_before_approval
    .withColumn(
        "hours_difference",
        (
            F.unix_timestamp("order_approved_at") -
            F.unix_timestamp("order_delivered_carrier_date")
        ) / 3600
    )
    .orderBy(F.desc("hours_difference"))
    .select(
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "hours_difference"
    )
    .limit(10)
)

order_id,order_status,order_approved_at,order_delivered_carrier_date,hours_difference
7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:50:22,2018-01-26 13:35:00,4109.256111111111
1fab4ac9d85079b3da72a11475ae1685,delivered,2017-09-13 22:06:11,2017-09-04 13:10:23,224.93
0184d4ddb259e1a4cfc2871888cf97b8,delivered,2017-09-13 22:17:15,2017-09-04 14:05:50,224.19027777777777
1378f9601350615613cc8832d6789c5d,delivered,2017-09-13 22:03:51,2017-09-04 18:07:55,219.9322222222222
8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-13 21:58:04,2017-09-04 19:12:19,218.7625
cf72398d0690f841271b695bbfda82d2,delivered,2017-09-13 22:04:39,2017-09-04 20:12:41,217.8661111111111
40de47dfa620d667117e4a6067b6e1ec,delivered,2017-09-13 21:58:38,2017-09-04 20:36:58,217.36111111111111
bc4854efd86d9f42140c951c595d20c1,delivered,2017-09-13 22:00:51,2017-09-04 20:49:57,217.18166666666667
77ca435b03fbf991e5027e3776e37885,delivered,2017-09-13 21:56:08,2017-09-04 20:50:00,217.10222222222222
580f3268bc2075c5961b2d2929c7a35b,delivered,2017-09-13 22:03:42,2017-09-05 11:58:18,202.09


* The primary key appears reliable.
* Missing timestamps are often related to order lifecycle status.
* Some timestamp sequences violate expected business logic.
* Some anomalies appear systematic.
* Bronze must preserve the original source data.

In [0]:
for file in dbutils.fs.ls(source_path):
    if file.name.endswith(".csv"):
        df = (
            spark.read
            .format("csv")
            .option("header", True)
            .option("inferSchema", False)
            .load(file.path)
        )

        print(f"\n{file.name}")
        print(df.columns)


olist_customers_dataset.csv
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_geolocation_dataset.csv
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

olist_order_items_dataset.csv
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset.csv
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_order_reviews_dataset.csv
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

olist_orders_dataset.csv
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_products_dataset.csv
['product_id', 'product_category_name', 'pro